# Instalacion de dependencias

In [65]:
!pip install python-dotenv psycopg2
!pip install supabase -q

# conexiones base de datos y supabase

In [66]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
from supabase import create_client

SUPABASE_URL = "https://ysbchbtuubrphiulkvlh.supabase.co"
SUPABASE_KEY = userdata.get('API_KEY')  # 👈 lee el secreto de Colab

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
CSV_PATH = "/content/drive/MyDrive/CSV"
print(" Listo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Listo


# Sincronizacion

In [67]:
# Sincronizar TODAS las categorías de products → product_category_name_translation
df_products = pd.read_csv(f"{CSV_PATH}/olist_products_dataset.csv")

# Traer las que ya existen en Supabase
existentes = supabase.table("product_category_name_translation").select("product_category_name").execute()
existentes_set = {r['product_category_name'] for r in existentes.data}

# Calcular faltantes
todas = set(df_products['product_category_name'].dropna().unique())
faltantes = todas - existentes_set

if faltantes:
    print(f"{len(faltantes)} categorías faltantes: {faltantes}")
    rows = [{"product_category_name": c, "product_category_name_english": None} for c in faltantes]
    supabase.table("product_category_name_translation").insert(rows).execute()
    print("Todas las categorías sincronizadas")
else:
    print("Todas las categorías ya existen")

Todas las categorías ya existen


In [70]:
# Sincronizar customers faltantes usando upsert
df_customers = pd.read_csv(f"{CSV_PATH}/olist_customers_dataset.csv")
df_customers = df_customers.where(pd.notnull(df_customers), None)
rows = df_customers.to_dict(orient="records")

print(f" Haciendo upsert de {len(rows)} customers...")
for i in range(0, len(rows), 500):
    supabase.table("customers").upsert(rows[i:i+500]).execute()
    print(f"  customers: {min(i+500, len(rows))}/{len(rows)}", end="\r")

print("\ Customers sincronizados")

📂 Haciendo upsert de 99441 customers...
  customers: 99441/99441
✅ Customers sincronizados


In [72]:
# Sincronizar sellers faltantes usando upsert
df_sellers = pd.read_csv(f"{CSV_PATH}/olist_sellers_dataset.csv")
df_sellers = df_sellers.where(pd.notnull(df_sellers), None)
rows = df_sellers.to_dict(orient="records")

print(f" Haciendo upsert de {len(rows)} sellers...")
for i in range(0, len(rows), 500):
    supabase.table("sellers").upsert(rows[i:i+500]).execute()
    print(f"  sellers: {min(i+500, len(rows))}/{len(rows)}", end="\r")
print("\ Sellers sincronizados")

📂 Haciendo upsert de 3095 sellers...
  sellers: 3095/3095
✅ Sellers sincronizados


In [74]:
# Insertar sellers huérfanos que no están en el CSV
df_order_items = pd.read_csv(f"{CSV_PATH}/olist_order_items_dataset.csv")
sellers_en_items   = set(df_order_items['seller_id'].dropna().unique())
sellers_en_csv     = set(df_sellers['seller_id'].dropna().unique())
sellers_huerfanos  = sellers_en_items - sellers_en_csv

if sellers_huerfanos:
    print(f"  {len(sellers_huerfanos)} sellers sin datos, insertando vacíos...")
    rows_vacios = [{"seller_id": s, "seller_zip_code_prefix": None,
                    "seller_city": None, "seller_state": None}
                   for s in sellers_huerfanos]
    for i in range(0, len(rows_vacios), 500):
        supabase.table("sellers").upsert(rows_vacios[i:i+500]).execute()
    print(" Sellers huérfanos insertados")

In [76]:
# Cargar order_reviews con upsert por duplicados en el CSV
df = pd.read_csv(f"{CSV_PATH}/olist_order_reviews_dataset.csv")
df = df.where(pd.notnull(df), None)

# Eliminar duplicados del CSV antes de insertar
df = df.drop_duplicates(subset=['review_id'])
print(f" Cargando order_reviews ({len(df)} filas sin duplicados)...")

rows = df.to_dict(orient="records")
for i in range(0, len(rows), 500):
    supabase.table("order_reviews").upsert(rows[i:i+500]).execute()
    print(f"  order_reviews: {min(i+500, len(rows))}/{len(rows)}", end="\r")
print("\n order_reviews completada")

📂 Cargando order_reviews (98410 filas sin duplicados)...
  order_reviews: 98410/98410
✅ order_reviews completada


In [77]:
df = pd.read_csv(f"{CSV_PATH}/olist_geolocation_dataset.csv")
df = df.where(pd.notnull(df), None)
print(f" Cargando geolocation ({len(df)} filas)...")

rows = df.to_dict(orient="records")
for i in range(0, len(rows), 500):
    supabase.table("geolocation").insert(rows[i:i+500]).execute()
    print(f"  geolocation: {min(i+500, len(rows))}/{len(rows)}", end="\r")
print("\n geolocation completada")

📂 Cargando geolocation (1000163 filas)...
  geolocation: 1000163/1000163
✅ geolocation completada
